In [5]:
!pip -q install openai datasets pandas scikit-learn tqdm tenacity

In [6]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
print("API key loaded securely.")

Enter your OpenAI API key: ··········
API key loaded securely.


In [8]:
import os
import json
import time
import pandas as pd

from json import JSONDecodeError
from tqdm import tqdm
from datasets import load_dataset
from openai import OpenAI, RateLimitError
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

MODEL = "gpt-5.5"
DATASET_NAME = "ChanceFocus/flare-causal20-sc"

# First test with 20 or 100.
# Then set LIMIT = None for the full test set.
LIMIT = 100

dataset = load_dataset(DATASET_NAME)

split_name = "test" if "test" in dataset else list(dataset.keys())[0]
split_data = dataset[split_name]

print(dataset)
print("Split:", split_name)
print("Columns:", split_data.column_names)
print("Example:", split_data[0])

VALID_LABELS = ["noise", "causal"]

CLASSIFICATION_SCHEMA = {
    "type": "object",
    "properties": {
        "label": {
            "type": "string",
            "enum": VALID_LABELS
        }
    },
    "required": ["label"],
    "additionalProperties": False
}

SYSTEM_PROMPT = """
You are a financial text classification model.

Task:
Classify the input sentence as either "causal" or "noise".

Definitions:
- causal: the sentence indicates a causal relationship between financial events, conditions, decisions, or outcomes.
- noise: the sentence does not indicate a clear causal relationship.

Rules:
1. Return exactly one label: "causal" or "noise".
2. Do not explain your answer.
3. Return only valid JSON following the schema.
"""

def normalize_label(x):
    x = str(x).strip().lower()

    if x in ["0", "noise"]:
        return "noise"

    if x in ["1", "causal"]:
        return "causal"

    return "noise"

def get_gold_label(row):
    if "answer" in row and row["answer"] is not None:
        return normalize_label(row["answer"])

    if "gold" in row and row["gold"] is not None:
        return normalize_label(row["gold"])

    if "label" in row and row["label"] is not None:
        return normalize_label(row["label"])

    raise ValueError("Cannot find gold label from row.")

def get_text(row):
    if "text" in row and row["text"] is not None:
        return row["text"]

    if "query" in row and row["query"] is not None:
        return row["query"]

    raise ValueError("Cannot find text from row.")

@retry(
    retry=retry_if_exception_type((RateLimitError, JSONDecodeError, ValueError)),
    wait=wait_exponential(multiplier=5, min=10, max=120),
    stop=stop_after_attempt(5)
)
def call_openai_causal20(text):
    response = client.responses.create(
        model=MODEL,
        input=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"Text:\n{text}"
            }
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "causal20_sc_classification",
                "schema": CLASSIFICATION_SCHEMA,
                "strict": True
            }
        },
        reasoning={
            "effort": "low"
        },
        max_output_tokens=300
    )

    raw_output = response.output_text

    if raw_output is None or raw_output.strip() == "":
        raise ValueError("Empty model output")

    try:
        parsed = json.loads(raw_output)
    except JSONDecodeError:
        print("Raw model output:")
        print(repr(raw_output))
        raise

    if "label" not in parsed:
        raise ValueError(f"Missing label field in model output: {parsed}")

    return parsed

eval_data = split_data if LIMIT is None else split_data.select(range(LIMIT))

rows = []
gold_labels = []
pred_labels = []

for idx, row in enumerate(tqdm(eval_data)):
    text = get_text(row)
    gold = get_gold_label(row)

    try:
        result = call_openai_causal20(text)
        pred = normalize_label(result.get("label", "noise"))
        error = ""
    except Exception as e:
        pred = "noise"
        error = str(e)
        print(f"Error at row {idx}: {e}")

    gold_labels.append(gold)
    pred_labels.append(pred)

    rows.append({
        "idx": idx,
        "id": row.get("id", ""),
        "text": text,
        "gold_label": gold,
        "predicted_label": pred,
        "correct": gold == pred,
        "error": error
    })

    # Slow down to reduce RateLimitError.
    time.sleep(5)

accuracy = accuracy_score(gold_labels, pred_labels)

macro_precision = precision_score(
    gold_labels,
    pred_labels,
    labels=VALID_LABELS,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    gold_labels,
    pred_labels,
    labels=VALID_LABELS,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    gold_labels,
    pred_labels,
    labels=VALID_LABELS,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    gold_labels,
    pred_labels,
    labels=VALID_LABELS,
    average="weighted",
    zero_division=0
)

print("\n===== GPT-5.5 FLARE-Causal20-SC Evaluation =====")
print(f"Dataset: {DATASET_NAME}")
print(f"Split: {split_name}")
print(f"Model: {MODEL}")
print(f"Samples evaluated: {len(eval_data)}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall: {macro_recall:.4f}")
print(f"Macro F1 Score: {macro_f1:.4f}")
print(f"Weighted F1 Score: {weighted_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        gold_labels,
        pred_labels,
        labels=VALID_LABELS,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
confusion_df = pd.DataFrame(
    confusion_matrix(gold_labels, pred_labels, labels=VALID_LABELS),
    index=[f"gold_{x}" for x in VALID_LABELS],
    columns=[f"pred_{x}" for x in VALID_LABELS]
)
print(confusion_df)

df = pd.DataFrame(rows)
df.to_csv("gpt55_flare_causal20_sc_evaluation_results.csv", index=False)

metrics = {
    "dataset": DATASET_NAME,
    "split": split_name,
    "model": MODEL,
    "samples_evaluated": len(eval_data),
    "accuracy": accuracy,
    "macro_precision": macro_precision,
    "macro_recall": macro_recall,
    "macro_f1_score": macro_f1,
    "weighted_f1_score": weighted_f1
}

with open("gpt55_flare_causal20_sc_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

num_errors = df["error"].astype(str).str.len().gt(0).sum()

print("\n===== Error Summary =====")
print(f"Failed samples: {num_errors}")
print(f"Total samples: {len(df)}")
print(f"Failure rate: {num_errors / len(df):.4f}")

pd.set_option("display.max_colwidth", 120)
df.head()

DatasetDict({
    test: Dataset({
        features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
        num_rows: 8628
    })
})
Split: test
Columns: ['id', 'query', 'answer', 'text', 'choices', 'gold']
Example: {'id': 'causal20sc0', 'query': "In this task, you are provided with sentences extracted from financial news and SEC data. Your goal is to classify each sentence into either 'causal' or 'noise' based on whether or not it indicates a causal relationship between financial events. Please return only the category 'causal' or 'noise'.\nText:  Third Democratic presidential debate  September 12, 2019 at 9:54 PM EDT - Updated September 13 at 12:54 AM  WASHINGTON (AP)  -  Ten Democrats seeking the presidency tripped over some details Thursday night as they sparred in a debate thick with policy and personal stories. Several made provocative accusations that President Donald Trump inspired the deadly shooting in El Paso, Texas, last month.\nAnswer:", 'answer': 'noise', 'text': ' 

100%|██████████| 100/100 [12:12<00:00,  7.32s/it]


===== GPT-5.5 FLARE-Causal20-SC Evaluation =====
Dataset: ChanceFocus/flare-causal20-sc
Split: test
Model: gpt-5.5
Samples evaluated: 100
Accuracy: 0.7600
Macro Precision: 0.5200
Macro Recall: 0.8788
Macro F1 Score: 0.4695
Weighted F1 Score: 0.8542

Classification Report:
              precision    recall  f1-score   support

       noise       1.00      0.76      0.86        99
      causal       0.04      1.00      0.08         1

    accuracy                           0.76       100
   macro avg       0.52      0.88      0.47       100
weighted avg       0.99      0.76      0.85       100


Confusion Matrix:
             pred_noise  pred_causal
gold_noise           75           24
gold_causal           0            1

===== Error Summary =====
Failed samples: 0
Total samples: 100
Failure rate: 0.0000


,idx,id,text,gold_label,predicted_label,correct,error
0,0,causal20sc0,"Third Democratic presidential debate September 12, 2019 at 9:54 PM EDT - Updated September 13 at 12:54 AM WASHING...",noise,causal,False,
1,1,causal20sc1,"On the policy front, Bernie Sanders claimed his approach to health care has a stamp of approval from everyone who s...",noise,noise,True,
2,2,causal20sc2,Joe Biden misrepresented recent history when he said the administration he served as vice president didn't put migr...,noise,noise,True,
3,3,causal20sc3,"Here's a look at some of the assertions in the third round of Democratic primary debates, the first to have all qua...",noise,noise,True,
4,4,causal20sc4,"It killed 22 people, and injured many more, we were not defeated by that. Nor were we defined by that.",noise,noise,True,
